In [3]:
"""
Bespoke generative decoders for trial-averaged Bernoulli features

Implements (A) Beta Naive Bayes on averaged probabilities x_ij in (0,1)
and (B) Beta-Binomial Naive Bayes if you know trial count T (uses k_ij ≈ round(T*x_ij)).

Outputs:
- evidence matrix Phi (same shape as X): phi_ij = log p(x_ij|y=1) - log p(x_ij|y=0)
- generative score s_i = logit(pi) + sum_j phi_ij
- optional PLS reduction on Phi + logistic regression in reduced space

Designed for large p (e.g. 40k neurons). Uses chunking and vectorization.
"""

import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression

from scipy.special import betaln, gammaln

# ============================================================
# DEMO: LOAD YOUR DATA + RUN TRUE vs PERM CV
# ============================================================


VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'

vit = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']
R   = np.load(NEURAL_PATH).T

top1 = np.argmax(vit, axis=1)
y = (top1 <= 397).astype(int)
X = np.clip(R.astype(float), 1e-6, 1 - 1e-6)

T=50

# ============================================================
# NUMERICS
# ============================================================

def _clip01(x, eps=1e-6):
    return np.clip(x, eps, 1.0 - eps)

def _logit(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)

def _beta_logpdf(x, a, b):
    # x: (n, p), a,b: (p,)
    # returns (n, p)
    return (a - 1.0) * np.log(x) + (b - 1.0) * np.log(1.0 - x) - betaln(a, b)

def _betabinom_logpmf(k, T, a, b):
    # k: (n, p) ints, a,b: (p,)
    # log [ C(T,k) * B(k+a, T-k+b) / B(a,b) ]
    # returns (n,p)
    k = k.astype(np.int64)
    logC = gammaln(T + 1) - gammaln(k + 1) - gammaln(T - k + 1)
    return logC + betaln(k + a, (T - k) + b) - betaln(a, b)


# ============================================================
# ESTIMATION: mu_jc and shared kappa
# ============================================================

def estimate_mu_per_class(X, y, a0=0.5, b0=0.5):
    """
    X: (n, p) in (0,1)
    y: (n,) in {0,1}
    Returns mu0, mu1: (p,)
    Uses a Beta(a0,b0) pseudo-count shrinkage on the mean:
      mu = (sum x + a0) / (n_c + a0 + b0)
    """
    X = _clip01(X)
    y = y.astype(int)
    idx0 = (y == 0)
    idx1 = (y == 1)
    n0 = max(int(idx0.sum()), 1)
    n1 = max(int(idx1.sum()), 1)

    s0 = X[idx0].sum(axis=0) if idx0.any() else np.zeros(X.shape[1])
    s1 = X[idx1].sum(axis=0) if idx1.any() else np.zeros(X.shape[1])

    mu0 = (s0 + a0) / (n0 + a0 + b0)
    mu1 = (s1 + a0) / (n1 + a0 + b0)
    return _clip01(mu0), _clip01(mu1)

def estimate_shared_kappa(X, y, mu0, mu1, kappa_min=2.0, kappa_max=1e6):
    """
    Shared concentration kappa using a pooled method-of-moments:
      Var[ X | class c ] ≈ mu_c(1-mu_c) / (kappa + 1)
    We estimate average empirical variance across neurons and classes,
    and solve for kappa.
    """
    X = _clip01(X)
    y = y.astype(int)
    idx0 = (y == 0)
    idx1 = (y == 1)

    # Empirical variances per neuron within each class (ddof=1 if possible)
    def safe_var(A):
        if A.shape[0] <= 1:
            return np.zeros(A.shape[1])
        return A.var(axis=0, ddof=1)

    v0 = safe_var(X[idx0]) if idx0.any() else np.zeros(X.shape[1])
    v1 = safe_var(X[idx1]) if idx1.any() else np.zeros(X.shape[1])

    # Expected beta variance numerator per class
    num0 = mu0 * (1.0 - mu0)
    num1 = mu1 * (1.0 - mu1)

    # Pool across classes and neurons robustly
    num = 0.5 * (num0 + num1)
    den = 0.5 * (v0 + v1)

    # Avoid dividing by ~0 variances (super-stable neurons)
    mask = den > np.percentile(den, 10)  # keep the more informative 90%
    if mask.sum() < 10:
        mask = den > 0

    if mask.sum() == 0:
        return 50.0  # fallback

    num_m = float(np.mean(num[mask]))
    den_m = float(np.mean(den[mask]))

    # kappa ≈ num/var - 1
    kappa = (num_m / max(den_m, 1e-12)) - 1.0
    kappa = float(np.clip(kappa, kappa_min, kappa_max))
    return kappa


# ============================================================
# GENERATIVE DECODERS
# ============================================================

class BetaNaiveBayesEvidence:
    """
    x_ij | y=c ~ Beta(alpha_jc, beta_jc)
    alpha_jc = mu_jc * kappa, beta_jc = (1-mu_jc)*kappa
    Shared kappa across neurons and classes (estimated from data unless provided).
    """

    def __init__(self, a0=0.5, b0=0.5, kappa=None, eps=1e-6, chunk=4096):
        self.a0 = a0
        self.b0 = b0
        self.kappa = kappa
        self.eps = eps
        self.chunk = chunk

        # fitted
        self.mu0_ = None
        self.mu1_ = None
        self.alpha0_ = None
        self.beta0_ = None
        self.alpha1_ = None
        self.beta1_ = None
        self.pi_ = None

    def fit(self, X, y):
        X = _clip01(np.asarray(X, float), self.eps)
        y = np.asarray(y, int)

        self.pi_ = float(np.mean(y))
        mu0, mu1 = estimate_mu_per_class(X, y, self.a0, self.b0)
        self.mu0_, self.mu1_ = mu0, mu1

        kappa = self.kappa
        if kappa is None:
            kappa = estimate_shared_kappa(X, y, mu0, mu1)
        self.kappa = float(kappa)

        self.alpha0_ = _clip01(mu0, self.eps) * self.kappa
        self.beta0_  = (1.0 - _clip01(mu0, self.eps)) * self.kappa
        self.alpha1_ = _clip01(mu1, self.eps) * self.kappa
        self.beta1_  = (1.0 - _clip01(mu1, self.eps)) * self.kappa
        return self

    def evidence_matrix(self, X):
        """
        Phi_ij = log p(x_ij|y=1) - log p(x_ij|y=0)
        Returns Phi of shape (n, p). Chunked over neurons for memory.
        """
        X = _clip01(np.asarray(X, float), self.eps)
        n, p = X.shape
        Phi = np.empty((n, p), dtype=np.float32)

        for j0 in range(0, p, self.chunk):
            j1 = min(p, j0 + self.chunk)
            Xc = X[:, j0:j1]
            ll1 = _beta_logpdf(Xc, self.alpha1_[j0:j1], self.beta1_[j0:j1])
            ll0 = _beta_logpdf(Xc, self.alpha0_[j0:j1], self.beta0_[j0:j1])
            Phi[:, j0:j1] = (ll1 - ll0).astype(np.float32)
        return Phi

    def score(self, X):
        """
        s_i = logit(pi) + sum_j phi_ij
        """
        Phi = self.evidence_matrix(X)
        return _logit(self.pi_) + Phi.sum(axis=1)

    def predict_proba(self, X):
        s = self.score(X)
        p1 = 1.0 / (1.0 + np.exp(-s))
        return np.vstack([1 - p1, p1]).T

    def predict(self, X, thresh=0.5):
        return (self.predict_proba(X)[:, 1] >= thresh).astype(int)


class BetaBinomialNaiveBayesEvidence:
    """
    If you know T (trials per image), use k_ij | y=c ~ BetaBinomial(T, alpha_jc, beta_jc)
    We take k_ij ≈ round(T * x_ij) if only x_ij is available.
    Same mu/kappa parameterization for stability.
    """

    def __init__(self, T, a0=0.5, b0=0.5, kappa=None, eps=1e-6, chunk=4096):
        self.T = int(T)
        self.a0 = a0
        self.b0 = b0
        self.kappa = kappa
        self.eps = eps
        self.chunk = chunk

        self.mu0_ = None
        self.mu1_ = None
        self.alpha0_ = None
        self.beta0_ = None
        self.alpha1_ = None
        self.beta1_ = None
        self.pi_ = None

    def fit(self, X, y):
        X = _clip01(np.asarray(X, float), self.eps)
        y = np.asarray(y, int)

        self.pi_ = float(np.mean(y))
        mu0, mu1 = estimate_mu_per_class(X, y, self.a0, self.b0)
        self.mu0_, self.mu1_ = mu0, mu1

        kappa = self.kappa
        if kappa is None:
            kappa = estimate_shared_kappa(X, y, mu0, mu1)
        self.kappa = float(kappa)

        self.alpha0_ = _clip01(mu0, self.eps) * self.kappa
        self.beta0_  = (1.0 - _clip01(mu0, self.eps)) * self.kappa
        self.alpha1_ = _clip01(mu1, self.eps) * self.kappa
        self.beta1_  = (1.0 - _clip01(mu1, self.eps)) * self.kappa
        return self

    def evidence_matrix(self, X):
        X = _clip01(np.asarray(X, float), self.eps)
        n, p = X.shape
        Phi = np.empty((n, p), dtype=np.float32)

        k = np.rint(self.T * X).astype(np.int64)  # approximate counts
        k = np.clip(k, 0, self.T)

        for j0 in range(0, p, self.chunk):
            j1 = min(p, j0 + self.chunk)
            kc = k[:, j0:j1]
            ll1 = _betabinom_logpmf(kc, self.T, self.alpha1_[j0:j1], self.beta1_[j0:j1])
            ll0 = _betabinom_logpmf(kc, self.T, self.alpha0_[j0:j1], self.beta0_[j0:j1])
            Phi[:, j0:j1] = (ll1 - ll0).astype(np.float32)
        return Phi

    def score(self, X):
        Phi = self.evidence_matrix(X)
        return _logit(self.pi_) + Phi.sum(axis=1)

    def predict_proba(self, X):
        s = self.score(X)
        from scipy.special import expit
        p1 = expit(s)
        return np.vstack([1 - p1, p1]).T

    def predict(self, X, thresh=0.5):
        return (self.predict_proba(X)[:, 1] >= thresh).astype(int)


# ============================================================
# PLS ON EVIDENCE + LOGISTIC
# ============================================================

def fit_pls_logistic(Phi_train, y_train, Phi_test, n_components=5, C=1.0, max_iter=2000):
    """
    PLSRegression learns supervised components of Phi for y.
    Then logistic regression is fit on the PLS scores.

    Returns: (yhat_test, proba_test, pls, clf, Z_train, Z_test)
    """
    # PLS wants y as float column
    pls = PLSRegression(n_components=n_components, scale=False)
    pls.fit(Phi_train, y_train.astype(float).reshape(-1, 1))

    Z_train = pls.transform(Phi_train)
    Z_test  = pls.transform(Phi_test)

    clf = LogisticRegression(penalty="l2", C=C, solver="lbfgs", max_iter=max_iter)
    clf.fit(Z_train, y_train)

    proba = clf.predict_proba(Z_test)[:, 1]
    yhat = (proba >= 0.5).astype(int)
    return yhat, proba, pls, clf, Z_train, Z_test

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from scipy.stats import ttest_rel, wilcoxon, binomtest

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------
# X : (n_images, n_neurons) in (0,1)
# y : (n_images,) in {0,1}
# T : number of trials per image (for Beta-Binomial)
#
# assumes your BetaBinomialNaiveBayesEvidence class is imported
# ------------------------------------------------------------

def loo_compare_logistic_vs_betabinom(X, y, T, C=1.0, random_state=0):
    n = X.shape[0]
    loo = LeaveOneOut()

    correct_log = np.zeros(n, dtype=int)
    correct_bb  = np.zeros(n, dtype=int)

    for fold, (train_idx, test_idx) in enumerate(loo.split(X)):
        Xtr, Xte = X[train_idx], X[test_idx]
        ytr, yte = y[train_idx], y[test_idx][0]

        # -------------------------
        # Logistic regression
        # -------------------------
        clf = LogisticRegression(
            penalty="l2",
            C=C,
            solver="lbfgs",
            max_iter=5000
        )
        clf.fit(Xtr, ytr)
        yhat_log = clf.predict(Xte)[0]
        correct_log[fold] = int(yhat_log == yte)

        # -------------------------
        # Beta-Binomial Naive Bayes
        # -------------------------
        bb = BetaBinomialNaiveBayesEvidence(T=T)
        bb.fit(Xtr, ytr)
        yhat_bb = bb.predict(Xte)[0]
        correct_bb[fold] = int(yhat_bb == yte)

    return correct_log, correct_bb


# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------

correct_log, correct_bb = loo_compare_logistic_vs_betabinom(X, y, T)

acc_log = correct_log.mean()
acc_bb  = correct_bb.mean()

print(f"LOO accuracy Logistic      : {acc_log:.3f}")
print(f"LOO accuracy Beta-Binomial : {acc_bb:.3f}")

# ------------------------------------------------------------
# PAIRED HYPOTHESIS TESTS
# ------------------------------------------------------------

diff = correct_log - correct_bb

# 1) Paired t-test (difference in means)
t_stat, p_t = ttest_rel(correct_log, correct_bb)

# 2) Wilcoxon signed-rank (nonparametric)
try:
    w_stat, p_w = wilcoxon(correct_log, correct_bb)
except ValueError:
    p_w = np.nan  # happens if all diffs are zero

# 3) Exact sign test (binomial)
n_pos = np.sum(diff > 0)
n_neg = np.sum(diff < 0)
n_eff = n_pos + n_neg

if n_eff > 0:
    p_sign = binomtest(
        k=min(n_pos, n_neg),
        n=n_eff,
        p=0.5,
        alternative="two-sided"
    ).pvalue
else:
    p_sign = 1.0

print("\n=== Paired hypothesis tests ===")
print(f"Paired t-test p-value      : {p_t:.4g}")
print(f"Wilcoxon signed-rank p     : {p_w:.4g}")
print(f"Exact sign test p          : {p_sign:.4g}")

# ------------------------------------------------------------
# AGREEMENT STRUCTURE (useful for thesis)
# ------------------------------------------------------------

both_correct = np.sum((correct_log == 1) & (correct_bb == 1))
both_wrong   = np.sum((correct_log == 0) & (correct_bb == 0))
log_only     = np.sum((correct_log == 1) & (correct_bb == 0))
bb_only      = np.sum((correct_log == 0) & (correct_bb == 1))

print("\n=== Error agreement ===")
print(f"Both correct : {both_correct}")
print(f"Both wrong   : {both_wrong}")
print(f"Logistic only correct : {log_only}")
print(f"BetaBinom only correct: {bb_only}")


LOO accuracy Logistic      : 0.695
LOO accuracy Beta-Binomial : 0.669

=== Paired hypothesis tests ===
Paired t-test p-value      : 0.4692
Wilcoxon signed-rank p     : 0.4669
Exact sign test p          : 0.6291

=== Error agreement ===
Both correct : 72
Both wrong   : 29
Logistic only correct : 10
BetaBinom only correct: 7
